# 03 · Stock trading costs and hedge-volatility error

**Objective:** measure how stock execution costs and an incorrect volatility used for hedge deltas change the three-name short-call portfolio's results.

The baseline contracts, quantities, **stock paths and opening premiums stay fixed**. The cost experiment changes only the stock trading charge. The hedge-volatility experiment changes only the volatility used for delta calculations, at a fixed trading charge. These experiments isolate implementation effects from a change in the option sale price.

The defaults use **5,000 correlated Q paths**, stock costs of **0, 1, 5 and 10 bp**, and hedge-volatility multipliers of **0.8, 1.0 and 1.2**. A final **10,000-path fresh-seed run** checks the pre-specified daily and five-session policies.

This completed notebook runs independently from a fresh kernel. Its model inputs load from the shared validated configuration used by notebook 02; experiment controls remain explicit. Inputs remain illustrative; quantities are share-equivalent short European call units, with zero dividends and constant financing. There is no exchange contract multiplier, option spread or margin constraint.

**Shared study inputs:** contract and scenario assumptions now load from `config/illustrative_study.json` through `load_named_inputs`. `Book` remains the validated model container. Change that file once to update the baseline across notebooks 00–03; policy/experiment controls stay explicit in each notebook. 


| Step | Purpose | Main functions/tools |
|---|---|---|
| 1–2 | Load packages and inspect named inputs | Dictionaries, `Book`, `initial_exposures` |
| 3 | Freeze the correlated baseline paths and premium | `simulate_paths`, source/input records |
| 4–5 | Vary stock costs and reconcile their financed effect | `hedge`, `summarize`, NumPy assertions |
| 6 | Compare daily and five-session means on matched paths | `paired_difference` |
| 7–8 | Vary hedge volatility while preserving option prices | `hedge(..., hedge_vols=...)` |
| 9 | Check fixed policies on a fresh larger sample | New seed, paired mean comparison |
| 10–11 | Interpret the results and record both runs | Calculated summaries, `provenance` |

All P&L statistics already include costs. Reporting those costs alongside net P&L does not mean subtracting them again. The paired intervals below concern **mean P&L differences**, not uncertainty in standard deviation or ES95.

## 1 · Connect to the project

Locate the project root and import the same pricing and accounting helpers used in the earlier notebooks. The kernel can start in either the project root or the notebook folder.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('03_cost_and_volatility_sensitivity')


## 2 · Define the named inputs and experiment controls

The portfolio and scenario load from the same validated study file as notebook 02. The controls specify the complete comparison before any results are calculated:

- Each policy opens a delta hedge and liquidates it at expiry. Initial-only holds its first stock hedge until settlement.
- Costs are one-way basis points of absolute stock dollars traded, including entry and liquidation.
- `REFERENCE_COST_BPS` sets the common charge for hedge-volatility sensitivity and fresh-seed checks.
- A hedge-volatility multiplier is a **relative change**: 0.8 means 20% below the base volatility, not 20 percentage points below it.
- `VALIDATION_POLICIES` are fixed here, before the fresh run. No policy is selected by optimizing its sample result.

This exercise uses matching pricing and path volatilities at baseline. Changing the option's pricing volatility or its true path volatility would be a separate experiment.

## 3 · Freeze the correlated paths and pricing inputs

Generate the same correlated baseline histories as notebook 02 using the same name order, inputs, dimensions and seed. Every baseline cost and hedge-volatility case receives this exact array.

A hash identifies the realized paths and is checked again after the experiments. The opening premium comes from the fixed book pricing volatilities. `hedge_vols` changes the stock hedge rule only; it does not change the sale premium or the accounting volatility used for interim option marks.

## 4 · Vary stock trading costs

Evaluate all four hedge policies at every charge in `COST_GRID_BPS`. Paths, contracts, pricing volatility and hedge volatility stay fixed. The same policy therefore trades the same stock quantities at every cost level; only cash and resulting net P&L change.

The table shows portfolio outcomes, computed from each path's summed name-level P&L. `mean_se` measures uncertainty in the mean; `pnl_std` describes economic variability. ES95 is minus the mean of the worst 5% of outcomes, rounded up to a whole observation count.

### Reconcile costs, trades and net P&L

For each policy, compare the costly result with its zero-cost counterpart on the same paths. For every outcome and name,

$$\text{zero-cost P\&L}-\text{costly P\&L}=\text{financed stock costs}.$$

Nominal costs add fees at their trade dates. Financed costs accumulate those amounts to expiry at the financing rate. The latter captures the complete terminal cash impact. Net P&L already includes that impact.

The checks also confirm identical turnover, trade counts, audited share holdings and opening premiums across cost levels.

## 5 · Compare risk against implementation cost

The charts show portfolio risk versus **mean financed stock cost**, matching the terminal-P&L accounting. Each point is labelled with its assumed basis-point charge. The tables retain initial-only; the charts focus on the three rebalanced policies because initial-only's much larger risk would compress their scale.

A point with lower tail risk can require more trading and a larger cost drag. No single point is universally best without a specified loss objective and budget. These curves are descriptive comparisons, rather than a policy optimization.

## 6 · Compare daily and five-session policies on matched paths

`paired_difference(daily, weekly)` computes the portfolio P&L difference **within each simulated outcome**, then estimates its mean and standard error. A negative mean means daily hedging produced lower net P&L on average in that sample.

The approximate interval is `mean difference ± 1.96 × paired SE`. It concerns the mean difference only. The risk-reduction columns report observed standard deviation and ES95 changes; the paired interval is not an interval for either risk statistic.

Costs can make mean P&L comparisons differ from risk comparisons: daily hedging can reduce downside variability while incurring more trading costs.

## 7 · Vary the volatility used for hedge deltas

Multiply each base pricing volatility by 0.8, 1.0 or 1.2 **only when calculating hedge delta**. For example, a base volatility of 22% becomes 17.6%, 22% or 26.4%.

Path volatility, contracts, quantities, rates, opening option premium and interim option-marking volatility remain fixed. This isolates hedge misspecification. An incorrect hedge volatility changes share holdings, residual risk, turnover and consequently costs; it does not create a different option sale price.

All cases use `REFERENCE_COST_BPS`. The multiplier-1 results reuse the existing cost experiment at that charge, ensuring an identical baseline.

## 8 · Verify the fixed-premium experiment and compare outcomes

For every volatility/policy case, check the opening premium, each audited option-liability mark, observed stock prices and expiry payoff against the multiplier-1 reference. Only the hedge holdings and their resulting cash flows are allowed to change.

The paired comparisons below subtract the **correct-volatility** result from the changed-volatility result on the same paths. A negative mean difference indicates lower average net P&L relative to the reference. ES95 differences are displayed separately and have no confidence interval here.

## 9 · Check fixed policies on a fresh larger sample

Generate 10,000 fresh outcomes with `VALIDATION_SEED` and the same contracts, marginal inputs and correlation matrix. Apply the daily and five-session rules specified in the input cell, at the reference cost and correct hedge volatility.

Within each run, the two policies share paths and their mean difference can be paired. The baseline and fresh runs are separate samples, so no pairing is applied across them. This checks simulation stability, rather than external market validity or an optimized policy choice.

The displayed mean intervals are approximate Monte Carlo intervals. No corresponding precision claim is made for ES95 or the volatility-sensitivity rankings.

### Visualize uncertainty in the paired mean difference

Each point is the estimated **daily minus five-session mean net P&L**. The horizontal interval is its approximate 95% Monte Carlo interval. This plot concerns the average outcome, while the preceding tables separately report the dispersion and tail losses of individual trading outcomes.

## 10 · Interpret the completed experiments

The calculated summary below reports how much daily hedging reduced risk relative to five-session hedging at the reference charge, and how its mean net P&L compared. The earlier cost and volatility tables show how those conclusions vary across the pre-specified assumptions.

A mean interval excluding zero identifies a difference large relative to its estimated Monte Carlo mean uncertainty in that comparison. It does not establish precision for ES95, economic optimality or a market trading edge.

Under the matched Q pricing/path model, a different predictable stock hedge changes risk and turnover without manufacturing a new option-pricing edge. With costs, expected net outcomes also reflect implementation drag. The experiments do not include real quotes, stochastic volatility, jumps, option spreads, dividends, margin or early exercise.

## 11 · Record both runs and all sensitivity assumptions

The baseline record includes the named portfolio, correlation scenario, cost grid, hedge-volatility grid, actual hedge volatilities, policies, audit path and realized-path hash. A nested validation record captures the separate seed, sample size, fixed policies and path hash used for the fresh run.

Runtime versions, source-file hashes and the notebook code hash make the implementation traceable. Hashing the paths again confirms that none of the hedge experiments modified the baseline or fresh price arrays. The record is displayed here without writing a separate export.

## What the completed study establishes

Notebook 00 defines and checks the model inputs and stock paths. Notebook 01 reconciles a single option's stock/cash ledger. Notebook 02 combines the three names and compares dependence and hedge frequency. This notebook adds stock costs, hedge-volatility errors and a fresh-sample check of fixed policies.

The cost identities reconcile path by path, and the hedge-volatility comparisons preserve the opening premiums, option marks, terminal payoffs and realized stock paths. Tables and figures distinguish risk reduction from cost drag and uncertainty in mean comparisons.

Future work can introduce dated market inputs, explicit policy objectives, fresh evaluation scenarios, stochastic volatility or jumps, and the market-accounting features required by a listed-option replay. The current results remain conditional on the illustrative Q-model assumptions.